# Credit Risk Scoring: End-to-End ML with snowflakeR

An end-to-end credit risk scoring workflow demonstrating how a fintech company
can build, evaluate, and deploy credit-risk models entirely within Snowflake
using R and the `snowflakeR` package.

**What you'll do:**
1. Explore lending data already in Snowflake
2. Build reusable features in the Snowflake Feature Store
3. Train competing models with tidymodels (logistic regression vs XGBoost)
4. Evaluate and compare model performance
5. Register both models in the Snowflake Model Registry with metrics
6. Select the champion version programmatically and deploy via SPCS

**Database layout** (created by the setup notebook):

| Schema | Contents |
|---|---|
| `RAW_DATA` | Source tables -- customers, payments, loans |
| `FEATURES` | Feature Store -- entity, feature views |
| `TRAINING` | Training datasets |
| `MODELS` | Model Registry -- versioned models, metrics |

**Migrating from Plumber?** The SPCS deployment gives you the same R model serving
capability as a Plumber API, but Snowflake handles scaling, authentication,
networking, and data proximity. Your R model code stays the same.

This notebook is for **Snowflake Workspace Notebooks** (Python kernel + `%%R` magic).

**Prerequisites:** Run `workspace_credit_risk_setup.ipynb` first to install the
R environment, create the database/schemas, and generate source data.

**Sections:**
1. Setup and Connect
2. Feature Store
3. Exploratory Data Analysis
4. Generate Training Data
5. Model Training (tidymodels)
6. Model Evaluation
7. Register Models and Select Champion
8. Deploy Champion and Inference
9. Cleanup

## 1. Setup and Connect

The R environment and tidymodels packages were installed by the setup notebook.
Here we configure rpy2, load packages, and connect to the data.

In [ ]:
from sfnb_setup import setup_notebook
setup_notebook(config="snowflaker_credit_risk_config.yaml")

In [ ]:
# Handled by setup_notebook() above

In [ ]:
%%R
library(snowflakeR)
library(tidymodels)
library(xgboost)
library(ggplot2)
library(dplyr)
library(scales)
rcat("tidymodels", as.character(packageVersion("tidymodels")), "loaded")
rcat("xgboost", as.character(packageVersion("xgboost")), "loaded")

### Connect and Configure Database Context

In [ ]:
%%R
conn <- sfr_connect()
conn <- sfr_load_notebook_config(conn, config_path = "snowflaker_credit_risk_config.yaml")

cr <- conn$notebook_config$credit_risk

cr_db       <- cr$database              %||% "CREDIT_RISK_ML"
cr_source   <- cr$source_schema         %||% "RAW_DATA"
cr_features <- cr$feature_store_schema  %||% "FEATURES"
cr_training <- cr$training_schema       %||% "TRAINING"
cr_registry <- cr$registry_schema       %||% "MODELS"

fqn_source  <- function(name) paste(cr_db, cr_source,  name, sep = ".")
fqn_model   <- function(name) paste(cr_db, cr_registry, name, sep = ".")

tbl_customers <- fqn_source("CREDIT_CUSTOMERS")
tbl_payments  <- fqn_source("CREDIT_PAYMENTS")
tbl_loans     <- fqn_source("CREDIT_LOANS")

n <- sfr_query(conn, paste("SELECT COUNT(*) AS n FROM", tbl_loans))
rcat(sprintf("Connected. %s loan applications in %s.",
             format(n$N, big.mark = ","), tbl_loans))
conn

### Model Registry

In [ ]:
%%R
reg <- sfr_model_registry(conn,
  database = cr_db,
  schema   = cr_registry
)
rcat(sprintf("Model Registry: %s.%s", cr_db, cr_registry))

## 2. Feature Store

Define reusable features in the **FEATURES** schema using `dplyr`/`dbplyr`
pipelines. Each Feature View is backed by a Snowflake Dynamic Table that
refreshes automatically.

Features defined here are used identically in training and production scoring --
**define once, use everywhere**.

`sfr_create_feature_view()` accepts either a **SQL string** or a **`dbplyr`
lazy table**. Below we demonstrate both: the first Feature View renders
`dbplyr` to a SQL string and passes that; the other two pass the `dbplyr`
lazy table directly. Either way, the pipeline runs entirely in Snowflake --
no data is pulled to the client.

In [ ]:
%%R
fs <- sfr_feature_store(conn,
  database  = cr_db,
  schema    = cr_features,
  warehouse = conn$warehouse,
  create    = TRUE
)

dbi_con <- sfr_dbi_connection(conn)

customers_tbl <- tbl(dbi_con, I(tbl_customers))
payments_tbl  <- tbl(dbi_con, I(tbl_payments))
loans_tbl     <- tbl(dbi_con, I(tbl_loans))

rcat("Feature Store and dbplyr table references ready.")
fs

### Create Entity

In [ ]:
%%R
customer_entity <- sfr_create_entity(
  fs, "CUSTOMER", "CUSTOMER_ID",
  desc = "Loan applicant"
)
customer_entity

### Feature View 1: Customer Profile

Static demographic features derived from the customer table.

We build the feature query as a `dbplyr` pipeline, then render it to a
**SQL string** with `dbplyr::sql_render()` and pass that SQL to
`sfr_create_feature_view()`.

`sfr_create_feature_view()` accepts either a **SQL string** or a
**`dbplyr` lazy table** -- the next two Feature Views demonstrate the
`dbplyr` approach directly.

In [ ]:
%%R
profile_query <- customers_tbl |>
  select(CUSTOMER_ID, AGE, ANNUAL_INCOME, EMPLOYMENT_LENGTH_YEARS,
         HOME_OWNERSHIP) |>
  mutate(LOG_INCOME = log(pmax(ANNUAL_INCOME, 1)))

profile_sql <- dbplyr::sql_render(profile_query)
rcat("dbplyr pipeline rendered to SQL:")
rcat(profile_sql)

In [ ]:
%%R
fv_profile <- sfr_create_feature_view(
  fs,
  name      = "CUSTOMER_PROFILE_FV",
  version   = "v1",
  entities  = customer_entity,
  features  = profile_sql,
  desc      = "Customer demographic profile features",
  overwrite = TRUE
)
rcat("Registered CUSTOMER_PROFILE_FV (using SQL string)")

### Feature View 2: Payment Behaviour (Dynamic Table)

Aggregated payment history features -- captures delinquency patterns.
Here we pass the **`dbplyr` lazy table directly** to `sfr_create_feature_view()`
(contrast with the SQL string approach used for FV1).

Payment data is transactional and arrives continuously, so this Feature View
uses `refresh_freq` to create a **Dynamic Table** that Snowflake refreshes
automatically. Use Dynamic Tables when source data changes frequently and
you want the Feature Store to keep features up-to-date without manual refreshes.

In [ ]:
%%R
payment_query <- payments_tbl |>
  group_by(CUSTOMER_ID) |>
  summarise(
    AVG_DAYS_PAST_DUE  = mean(DAYS_PAST_DUE, na.rm = TRUE),
    MAX_DAYS_PAST_DUE  = max(DAYS_PAST_DUE, na.rm = TRUE),
    PCT_MISSED         = sum(ifelse(PAYMENT_STATUS == "MISSED", 1, 0),
                             na.rm = TRUE) / n(),
    PCT_ON_TIME        = sum(ifelse(PAYMENT_STATUS == "ON_TIME", 1, 0),
                             na.rm = TRUE) / n(),
    AVG_PAYMENT_AMOUNT = mean(PAYMENT_AMOUNT, na.rm = TRUE),
    TOTAL_PAYMENTS     = n(),
    .groups = "drop"
  )

fv_payments <- sfr_create_feature_view(
  fs,
  name         = "PAYMENT_BEHAVIOR_FV",
  version      = "v1",
  entities     = customer_entity,
  features     = payment_query,
  refresh_freq = "1 hour",
  desc         = "Payment behaviour aggregates (Dynamic Table)",
  overwrite    = TRUE
)
rcat("Registered PAYMENT_BEHAVIOR_FV (Dynamic Table, refresh = 1 hour)")

### Feature View 3: Credit History

Aggregated lending history features -- captures borrowing patterns. Also uses
the **`dbplyr` lazy table** approach. Loan applications change less frequently
than payments, so a regular View is sufficient here.

In [ ]:
%%R
credit_query <- loans_tbl |>
  group_by(CUSTOMER_ID) |>
  summarise(
    TOTAL_APPLICATIONS = n(),
    AVG_LOAN_AMOUNT    = mean(LOAN_AMOUNT, na.rm = TRUE),
    MAX_LOAN_AMOUNT    = max(LOAN_AMOUNT, na.rm = TRUE),
    PRIOR_DEFAULTS     = sum(DEFAULTED, na.rm = TRUE),
    AVG_INTEREST_RATE  = mean(INTEREST_RATE, na.rm = TRUE),
    .groups = "drop"
  )

fv_credit <- sfr_create_feature_view(
  fs,
  name      = "CREDIT_HISTORY_FV",
  version   = "v1",
  entities  = customer_entity,
  features  = credit_query,
  desc      = "Credit / lending history aggregates",
  overwrite = TRUE
)
rcat("Registered CREDIT_HISTORY_FV")

In [ ]:
%%R
rcat("Feature Views registered:")
sfr_list_feature_views(fs)

## 3. Exploratory Data Analysis

Quick look at the data distributions and relationships with default.


In [ ]:
%%R
eda_query <- loans_tbl |>
  left_join(
    customers_tbl |>
      select(CUSTOMER_ID, AGE, ANNUAL_INCOME,
             EMPLOYMENT_LENGTH_YEARS, HOME_OWNERSHIP),
    by = "CUSTOMER_ID"
  )

rcat("Generated SQL:")
rcat(dbplyr::sql_render(eda_query))
rcat("")

loans_df <- collect(eda_query)
loans_df$DEFAULTED <- factor(loans_df$DEFAULTED, levels = c(0, 1),
                                labels = c("No Default", "Default"))
rcat(sprintf("Joined dataset: %d rows x %d cols", nrow(loans_df), ncol(loans_df)))

### Default Rate Overview


In [ ]:
%%R -w 900 -h 350
suppressWarnings(print(
  ggplot(loans_df, aes(x = DEFAULTED, fill = DEFAULTED)) +
    geom_bar(show.legend = FALSE) +
    geom_text(stat = "count", aes(label = comma(after_stat(count))),
              vjust = -0.5, size = 4) +
    scale_fill_manual(values = c("#2ecc71", "#e74c3c")) +
    scale_y_continuous(labels = comma, expand = expansion(mult = c(0, 0.1))) +
    labs(title = "Loan Default Distribution",
         subtitle = "~15% default rate -- realistic class imbalance",
         x = NULL, y = "Applications") +
    theme_minimal(base_size = 13) +
    theme(plot.title = element_text(face = "bold"))
))

### Income vs Default


In [ ]:
%%R -w 900 -h 400
suppressWarnings(print(
  ggplot(loans_df, aes(x = ANNUAL_INCOME, fill = DEFAULTED)) +
    geom_density(alpha = 0.5) +
    scale_x_continuous(labels = dollar) +
    coord_cartesian(xlim = c(0, 250000)) +
    scale_fill_manual(values = c("#2ecc71", "#e74c3c")) +
    labs(title = "Income Distribution by Default Status",
         subtitle = "Lower income correlates with higher default probability",
         x = "Annual Income", y = "Density", fill = NULL) +
    theme_minimal(base_size = 13) +
    theme(plot.title = element_text(face = "bold"),
          legend.position = "top")
))

### Employment Length vs Default Rate


In [ ]:
%%R -w 900 -h 400
emp_summary <- loans_df |>
  mutate(emp_bin = cut(EMPLOYMENT_LENGTH_YEARS,
                       breaks = c(-1, 2, 5, 10, 20, 50),
                       labels = c("0-2yr", "3-5yr", "6-10yr",
                                  "11-20yr", "20+yr"))) |>
  group_by(emp_bin) |>
  summarise(default_rate = mean(DEFAULTED == "Default"),
            n = n(), .groups = "drop")

suppressWarnings(print(
  ggplot(emp_summary, aes(x = emp_bin, y = default_rate)) +
    geom_col(fill = "steelblue") +
    geom_text(aes(label = sprintf("%.0f%%", default_rate * 100)),
              vjust = -0.5, size = 4) +
    scale_y_continuous(labels = percent, expand = expansion(mult = c(0, 0.15))) +
    labs(title = "Default Rate by Employment Length",
         subtitle = "Longer employment = lower default risk",
         x = "Employment Length", y = "Default Rate") +
    theme_minimal(base_size = 13) +
    theme(plot.title = element_text(face = "bold"))
))

## 4. Generate Training Data from Feature Store

Join the label spine (loan applications with default outcome) with all three
Feature Views. The Feature Store handles the joins automatically -- the same
features used here will be retrieved at inference time for production scoring.

The spine query references the source schema; Feature Views resolve from
the Feature Store schema.

In [ ]:
%%R
training_data <- sfr_generate_training_data(
  fs,
  spine = paste(
    "SELECT application_id, customer_id, defaulted",
    "FROM", tbl_loans
  ),
  features = list(fv_profile, fv_payments, fv_credit),
  spine_label_cols = "DEFAULTED"
)

tbl_training <- paste(cr_db, cr_training, "CREDIT_RISK_TRAINING", sep = ".")
sfr_write_table(conn, tbl_training, training_data, overwrite = TRUE)

rcat(sprintf("Training data: %d rows x %d cols -> %s",
             nrow(training_data), ncol(training_data), tbl_training))
head(as.data.frame(training_data), 5)

In [ ]:
%%R
training_data$DEFAULTED <- factor(training_data$DEFAULTED,
                                  levels = c(1, 0),
                                  labels = c("yes", "no"))

rcat("Label distribution:")
table(training_data$DEFAULTED)

## 5. Model Training with tidymodels

We train two models and compare them with cross-validation:
- **Logistic Regression** -- interpretable baseline, standard in credit risk
- **XGBoost** -- gradient-boosted trees, often better predictive performance

Both share the same preprocessing recipe.

### Preprocessing Recipe


In [ ]:
%%R
credit_recipe <- recipe(
  DEFAULTED ~ AGE + ANNUAL_INCOME + LOG_INCOME +
    EMPLOYMENT_LENGTH_YEARS + HOME_OWNERSHIP +
    AVG_DAYS_PAST_DUE + MAX_DAYS_PAST_DUE +
    PCT_MISSED + PCT_ON_TIME + AVG_PAYMENT_AMOUNT +
    TOTAL_PAYMENTS + TOTAL_APPLICATIONS + AVG_LOAN_AMOUNT +
    MAX_LOAN_AMOUNT + PRIOR_DEFAULTS + AVG_INTEREST_RATE,
  data = training_data
) |>
  step_dummy(HOME_OWNERSHIP) |>
  step_impute_median(all_numeric_predictors()) |>
  step_zv(all_predictors()) |>
  step_normalize(all_numeric_predictors())

credit_recipe

### Define Model Specifications


In [ ]:
%%R
logistic_spec <- logistic_reg() |>
  set_engine("glm") |>
  set_mode("classification")

xgb_spec <- boost_tree(
  trees = 500,
  tree_depth = 6,
  learn_rate = 0.05,
  min_n = 10
) |>
  set_engine("xgboost") |>
  set_mode("classification")

rcat("Model specifications defined")

### Create Workflows


In [ ]:
%%R
wf_logistic <- workflow() |>
  add_recipe(credit_recipe) |>
  add_model(logistic_spec)

wf_xgb <- workflow() |>
  add_recipe(credit_recipe) |>
  add_model(xgb_spec)

rcat("Workflows created")

### Cross-Validated Evaluation

5-fold stratified CV to get honest performance estimates.


In [ ]:
%%R
set.seed(42)
folds <- vfold_cv(training_data, v = 5, strata = DEFAULTED)

credit_metrics <- metric_set(roc_auc, accuracy, pr_auc)

rcat("Evaluating logistic regression (5-fold CV)...")
results_lr <- fit_resamples(wf_logistic, folds, metrics = credit_metrics)

rcat("Evaluating XGBoost (5-fold CV)...")
results_xgb <- fit_resamples(wf_xgb, folds, metrics = credit_metrics)

rcat("Cross-validation complete")

## 6. Model Evaluation

### Compare CV Metrics


In [ ]:
%%R
metrics_lr  <- collect_metrics(results_lr) |> mutate(model = "Logistic Regression")
metrics_xgb <- collect_metrics(results_xgb) |> mutate(model = "XGBoost")

comparison <- bind_rows(metrics_lr, metrics_xgb) |>
  select(model, .metric, mean, std_err) |>
  arrange(.metric, desc(mean))

as.data.frame(comparison)

In [ ]:
%%R -w 900 -h 400
suppressWarnings(print(
  ggplot(comparison, aes(x = .metric, y = mean, fill = model)) +
    geom_col(position = position_dodge(width = 0.7), width = 0.6) +
    geom_errorbar(aes(ymin = mean - std_err, ymax = mean + std_err),
                  position = position_dodge(width = 0.7), width = 0.2) +
    geom_text(aes(label = sprintf("%.3f", mean)),
              position = position_dodge(width = 0.7), vjust = -0.8, size = 3.5) +
    scale_fill_manual(values = c("#3498db", "#e67e22")) +
    scale_y_continuous(expand = expansion(mult = c(0, 0.15))) +
    labs(title = "Model Comparison: 5-Fold Cross-Validation",
         x = "Metric", y = "Mean Score", fill = NULL) +
    theme_minimal(base_size = 13) +
    theme(plot.title = element_text(face = "bold"),
          legend.position = "top")
))

**Reading the metrics:**

- **accuracy** -- overall correct predictions. Both models score ~75%, which is
  decent for a 15% default rate (a naive "predict no default" baseline would
  score 85%, but miss every actual default).
- **roc_auc** -- how well the model ranks defaulters above non-defaulters,
  across all possible thresholds. Values above 0.7 indicate useful
  discrimination; above 0.8 is good.
- **pr_auc** (Precision-Recall AUC) -- measures ranking quality with a focus
  on the **minority class** (defaults). PR-AUC is always lower than ROC-AUC
  when classes are imbalanced because it does not credit the model for
  correctly predicting the abundant "no default" class. A random classifier
  scores ~0.5 on ROC-AUC but only ~0.15 on PR-AUC (the base default rate).
  Our PR-AUC of 0.5-0.6 is well above that baseline, confirming the models
  have learned real signal.

### Train Final Models and Plot ROC Curves

Fit both models on the full training set and compare ROC curves on a holdout.


In [ ]:
%%R
set.seed(42)
split <- initial_split(training_data, prop = 0.8, strata = DEFAULTED)
train_set <- training(split)
test_set  <- testing(split)

fit_lr  <- fit(wf_logistic, data = train_set)
fit_xgb <- fit(wf_xgb, data = train_set)

preds_lr  <- augment(fit_lr, test_set)
preds_xgb <- augment(fit_xgb, test_set)

roc_lr  <- roc_curve(preds_lr, DEFAULTED, .pred_yes) |> mutate(model = "Logistic")
roc_xgb <- roc_curve(preds_xgb, DEFAULTED, .pred_yes) |> mutate(model = "XGBoost")

holdout_lr <- bind_rows(
  roc_auc(preds_lr, DEFAULTED, .pred_yes),
  pr_auc(preds_lr, DEFAULTED, .pred_yes),
  accuracy(preds_lr, DEFAULTED, .pred_class)
) |> mutate(model = "Logistic Regression")

holdout_xgb <- bind_rows(
  roc_auc(preds_xgb, DEFAULTED, .pred_yes),
  pr_auc(preds_xgb, DEFAULTED, .pred_yes),
  accuracy(preds_xgb, DEFAULTED, .pred_class)
) |> mutate(model = "XGBoost")

rcat("Holdout metrics:")
as.data.frame(bind_rows(holdout_lr, holdout_xgb))

In [ ]:
%%R -w 900 -h 500
roc_data <- bind_rows(roc_lr, roc_xgb)

auc_lr  <- roc_auc(preds_lr, DEFAULTED, .pred_yes)$.estimate
auc_xgb <- roc_auc(preds_xgb, DEFAULTED, .pred_yes)$.estimate

suppressWarnings(print(
  ggplot(roc_data, aes(x = 1 - specificity, y = sensitivity, color = model)) +
    geom_line(linewidth = 1.1) +
    geom_abline(linetype = "dashed", color = "grey50") +
    scale_color_manual(values = c("#3498db", "#e67e22"),
      labels = c(sprintf("Logistic (AUC=%.3f)", auc_lr),
                 sprintf("XGBoost (AUC=%.3f)", auc_xgb))) +
    labs(title = "ROC Curves: Logistic Regression vs XGBoost",
         x = "False Positive Rate", y = "True Positive Rate", color = NULL) +
    theme_minimal(base_size = 13) +
    theme(plot.title = element_text(face = "bold"),
          legend.position = c(0.7, 0.25),
          legend.background = element_rect(fill = "white", color = "grey80"))
))

**Reading the ROC curves:**

Each curve shows the trade-off between True Positive Rate (catching defaults)
and False Positive Rate (false alarms) as the decision threshold varies. The
dashed diagonal is a random classifier (AUC = 0.5); curves further toward the
top-left corner are better.

The logistic regression curve is smoother because it outputs a continuous
probability distribution. XGBoost's curve has visible steps because its 500
trees produce fewer unique probability values -- many applicants get bucketed
into the same predicted risk score, so the curve jumps in larger increments.
This is normal behaviour for tree ensembles and does not indicate a problem.

### Local Best-Model Summary

Based on local holdout evaluation, the model with the higher ROC-AUC is the
likely champion. We'll register **both** models in the next section and let the
Model Registry confirm this selection.


In [ ]:
%%R
if (auc_xgb >= auc_lr) {
  best_fit  <- fit_xgb
  best_name <- "XGBoost"
  best_auc  <- auc_xgb
  best_pkgs <- c("tidymodels", "xgboost")
} else {
  best_fit  <- fit_lr
  best_name <- "Logistic Regression"
  best_auc  <- auc_lr
  best_pkgs <- c("tidymodels")
}
rcat(sprintf("Best model: %s (AUC = %.3f)", best_name, best_auc))

## 7. Register Models and Select Champion

Register **both** candidate models as versions of `SFR_CREDIT_RISK_SCORER` in
the Model Registry, attach holdout metrics to each version, then
programmatically select the champion based on registry-stored metrics.

This mirrors a production workflow where the registry is the single source of
truth for model governance -- not local R variables.

**Version pinning:** `sfr_log_model()` automatically pins `r-base` and all
tidymodels sub-package versions to match this Workspace session
(`pin_versions = TRUE` by default). This ensures the SPCS inference container
uses identical package versions, preventing `hardhat::forge()` deserialization
failures caused by version drift.

In [ ]:
%%R
sfr_predict_local(best_fit, test_set[1:5, ])

In [ ]:
%%R
input_cols <- sfr_input_cols(
  training_data,
  exclude = c("DEFAULTED", "CUSTOMER_ID", "APPLICATION_ID")
)
rcat(sprintf("Inferred %d input columns from training data", length(input_cols)))

# Additional conda deps beyond what sfr_log_model auto-pins.
# R version + tidymodels sub-packages are pinned automatically
# from the current session (pin_versions = TRUE by default).
extra_deps <- c("r-xgboost", "r-ranger", "numpy<2.0")

mv_lr <- sfr_log_model(
  reg,
  model        = fit_lr,
  model_name   = "SFR_CREDIT_RISK_SCORER",
  predict_pkgs = c("tidymodels"),
  input_cols   = input_cols,
  output_cols  = list(prediction = "string"),
  conda_deps   = extra_deps,
  comment      = sprintf("Logistic regression -- AUC %.3f", auc_lr)
)

mv_xgb <- sfr_log_model(
  reg,
  model        = fit_xgb,
  model_name   = "SFR_CREDIT_RISK_SCORER",
  predict_pkgs = c("tidymodels", "xgboost"),
  input_cols   = input_cols,
  output_cols  = list(prediction = "string"),
  conda_deps   = extra_deps,
  comment      = sprintf("XGBoost -- AUC %.3f", auc_xgb)
)

rcat(sprintf("Registered versions: %s (LR), %s (XGB)",
             mv_lr$version_name, mv_xgb$version_name))

In [ ]:
%%R
for (ver in list(list(mv = mv_lr,  h = holdout_lr),
                 list(mv = mv_xgb, h = holdout_xgb))) {
  for (i in seq_len(nrow(ver$h))) {
    sfr_set_model_metric(reg, "SFR_CREDIT_RISK_SCORER",
                         ver$mv$version_name,
                         ver$h$.metric[i], ver$h$.estimate[i])
  }
  rcat(sprintf("  %s: attached %d metrics",
               ver$mv$version_name, nrow(ver$h)))
}
rcat("All metrics stored in Model Registry")

In [ ]:
%%R
versions <- sfr_show_model_versions(reg, "SFR_CREDIT_RISK_SCORER")

metrics_list <- Filter(Negate(is.null), lapply(versions$name, function(vn) {
  m <- sfr_show_model_metrics(reg, "SFR_CREDIT_RISK_SCORER", vn)
  if (length(m) == 0 || is.null(m$roc_auc)) return(NULL)
  data.frame(version = vn,
             roc_auc  = as.numeric(m$roc_auc),
             pr_auc   = as.numeric(m$pr_auc),
             accuracy = as.numeric(m$accuracy))
}))

registry_comparison <- do.call(rbind, metrics_list)
rcat("Model versions and metrics from registry:")
registry_comparison

### Select Champion Version

With metrics stored in the registry, we can programmatically select the best
version. Here we pick the version with the highest `roc_auc` and set it as the
model's **default version** -- the version that downstream consumers (services,
SQL functions, dashboards) will use automatically.

In [ ]:
%%R
champion_idx <- which.max(registry_comparison$roc_auc)
champion_ver <- registry_comparison$version[champion_idx]

sfr_set_default_model_version(reg, "SFR_CREDIT_RISK_SCORER", champion_ver)

rcat(sprintf("Champion: %s (roc_auc = %.3f, pr_auc = %.3f, accuracy = %.3f)",
             champion_ver,
             registry_comparison$roc_auc[champion_idx],
             registry_comparison$pr_auc[champion_idx],
             registry_comparison$accuracy[champion_idx]))
rcat(sprintf("Set as default version in Model Registry"))

## 8. Deploy Champion and Inference

Deploy the **champion version** as a scalable REST service via Snowpark
Container Services.

**Plumber comparison:** This replaces your Plumber API deployment. Same R model,
but Snowflake handles container orchestration, auto-scaling, authentication, and
data stays within the platform -- no data egress needed.

In [ ]:
%%R
sfr_create_compute_pool(conn, "R_CREDIT_POOL",
                        instance_family = "CPU_X64_M")
sfr_create_image_repo(conn, fqn_model("R_CREDIT_IMAGES"))

svc_fqn <- paste(cr_db, cr_registry, "CREDIT_RISK_SVC", sep = ".")
sfr_execute(conn, paste("DROP SERVICE IF EXISTS", svc_fqn))

sfr_deploy_model(
  reg,
  model_name   = "SFR_CREDIT_RISK_SCORER",
  version_name = champion_ver,
  service_name = "credit_risk_svc",
  compute_pool = "R_CREDIT_POOL",
  image_repo   = fqn_model("R_CREDIT_IMAGES"),
  force        = TRUE
)

In [ ]:
%%R
sfr_wait_for_service(reg, "credit_risk_svc",
                     timeout_min = 10, poll_sec = 15)

### Score New Applications

Retrieve features from the Feature Store for a batch of customers and
score them through the deployed model -- the same flow a production
pipeline would use.

In [ ]:
%%R
scoring_features <- sfr_retrieve_features(
  fs,
  spine = paste(
    "SELECT DISTINCT customer_id FROM", tbl_customers,
    "LIMIT 20"
  ),
  features = list(fv_profile, fv_payments, fv_credit)
)

rcat(sprintf("Retrieved %d rows x %d cols from Feature Store",
             nrow(scoring_features), ncol(scoring_features)))
scoring_features

In [ ]:
%%R
predictions <- sfr_predict(
  reg, "SFR_CREDIT_RISK_SCORER",
  scoring_features[, names(input_cols)],
  version_name = champion_ver,
  service_name = "credit_risk_svc"
)

results <- cbind(
  scoring_features[, "CUSTOMER_ID", drop = FALSE],
  predictions
)
rcat("Predictions for 20 customers:")
results

### Score from SQL

The deployed SPCS service exposes a `PREDICT` function callable directly from SQL,
making the model accessible to any Snowflake user, BI tool, or scheduled task --
no R or Python required.

In [ ]:
%%R
svc_fn   <- paste(cr_db, cr_registry, "CREDIT_RISK_SVC!PREDICT", sep = ".")
col_list <- paste(names(input_cols), collapse = ", ")

sql_score <- paste(
  "SELECT *,", svc_fn, "(", col_list, ") AS prediction",
  "FROM", paste(cr_db, cr_training, "CREDIT_RISK_TRAINING", sep = "."),
  "LIMIT 10"
)

rcat("SQL prediction query:")
rcat(sql_score)
rcat("")
sfr_query(conn, sql_score)

## 9. Cleanup

Uncomment to remove model, infrastructure, and Feature Store resources.
Source data tables and schemas are managed by the setup notebook.

In [ ]:
# Cleanup: SPCS services must be dropped BEFORE deleting the model.
# Uncomment and run the blocks below as needed.

# -- Drop all SPCS services in the registry schema --
# from snowflake.snowpark.context import get_active_session
# _sess = get_active_session()
# _svc_rows = _sess.sql(
#     f"SHOW SERVICES IN SCHEMA {cr_db}.{cr_registry}"
# ).collect()
# for _row in _svc_rows:
#     _name = _row["name"]
#     print(f"Dropping service: {_name}")
#     _sess.sql(f'DROP SERVICE IF EXISTS {cr_db}.{cr_registry}."{_name}"').collect()
# print("All services dropped.")

# -- Delete model (only after services are dropped) --
# _sess.sql(
#     f"DROP MODEL IF EXISTS {cr_db}.{cr_registry}.SFR_CREDIT_RISK_SCORER"
# ).collect()
# print("Model deleted.")

# -- Infrastructure cleanup --
# _sess.sql("DROP COMPUTE POOL IF EXISTS R_CREDIT_POOL").collect()
# _sess.sql(
#     f"DROP IMAGE REPOSITORY IF EXISTS {cr_db}.{cr_registry}.R_CREDIT_IMAGES"
# ).collect()
# print("Compute pool and image repository dropped.")

print("Cleanup section -- uncomment blocks above to delete resources")

## Summary

This notebook demonstrated a complete credit risk scoring workflow in Snowflake
with **schema-level isolation** for each concern:

| Schema | What we did |
|---|---|
| `RAW_DATA` | Queried source lending data |
| `FEATURES` | Created entity and three Feature Views |
| `TRAINING` | Generated training data from Feature Store joins |
| `MODELS` | Registered both models, compared metrics, deployed champion |

### Key Advantages Over Current Workflow

| Current (RStudio + Plumber) | Snowflake (snowflakeR + SPCS) |
|---|---|
| Data exported to local machine | Data stays in Snowflake |
| Manual feature engineering scripts | Feature Store: defined once, used everywhere |
| Model files on shared drives | Model Registry: versioned, with metrics |
| Plumber API on a VM | SPCS: auto-scaled, authenticated, governed |
| Separate monitoring tools | Snowflake ML Observability (drift detection) |

### Next Steps

- Add hyperparameter tuning with `tune::tune_grid()`
- Explore model monitoring with Snowflake ML Observability
- Schedule periodic retraining with Snowflake Tasks
- Add more Feature Views (e.g., bureau data, employment verification)
- See `workspace_model_registry.ipynb` for the full Model Registry API tour
- See `workspace_feature_store.ipynb` for advanced Feature Store patterns